# Optimization Basics Lab: Python companion
Interactive companion to Chapter 1 of [*Optimization Algorithms: AI techniques for design, planning, and control problems*](https://www.manning.com/books/optimization-algorithms) by Alaa Khamis (Manning, 2024).

These from-scratch implementations mirror the simulations in the Optimization Basics Lab. They need only the Python standard library.

## 1. Feasible solutions, global and local minima
The multimodal function of Figure 1.3 inside the bounds [-3, 3].

In [ ]:
# Feasible solutions, global and local minima (Section 1.3, Figure 1.3)
import math

f = lambda x: 3 * math.cos(x * x) - math.sin(3 * x) * math.exp(-x)    # the multimodal function of MathPlots.ipynb
LB, UB = -3, 3                                                       # boundary constraints on the decision variable


def local_minima(lo, hi, n=60000):
    """Scan a fine grid and keep the points lower than both neighbors."""
    xs = [lo + (hi - lo) * i / n for i in range(n + 1)]
    return [x for a, x, b in zip(xs, xs[1:], xs[2:]) if f(x) < f(a) and f(x) < f(b)]


minima = local_minima(-4, 4)
feasible = [x for x in minima if LB <= x <= UB]
best = min(feasible + [LB, UB], key=f)                               # the bounds can hold the optimum too
for x in minima:
    kind = "infeasible" if not LB <= x <= UB else "global minimum" if x == best else "strong local minimum"
    print(f"x = {x:6.3f}   f(x) = {f(x):8.3f}   {kind}")
print(f"Lowest point overall: x = {min(minima, key=f):.3f}, but it lies outside the bounds [{LB}, {UB}]")

## 2. Ticket pricing
Derivative, golden-section and grid search for the best price, plus the duality transformations of Figure 1.4.

In [ ]:
# The ticket pricing problem (Sections 1.3.1 and 1.3.2, Figures 1.4 and 1.5)
a, b = 50_000, 60                     # fixed costs ($) and variable cost per attendee ($)
LB, UB = 0, 250                       # boundary constraints on the ticket price
Q = lambda x: 5000 - 20 * x           # expected tickets sold at price x (equation 1.2)
profit = lambda x: Q(x) * x - (a + Q(x) * b)   # revenue minus costs = -20x^2 + 6,200x - 350,000


def golden_section(g, lo, hi, tol=1e-6):
    """Derivative-free search for the maximum of a unimodal function on [lo, hi]."""
    r = (5 ** 0.5 - 1) / 2
    c, d = hi - r * (hi - lo), lo + r * (hi - lo)
    while hi - lo > tol:
        if g(c) > g(d):
            hi, d = d, c
            c = hi - r * (hi - lo)
        else:
            lo, c = c, d
            d = lo + r * (hi - lo)
    return (lo + hi) / 2


x_star = 125 + b / 2                  # from df/dx = -40x + 5,000 + 20b = 0
print(f"Derivative: x* = ${x_star:.2f}, profit = ${profit(x_star):,.0f}")
x_gs = golden_section(profit, LB, UB)
print(f"Golden-section search: x* = ${x_gs:.2f}, profit = ${profit(x_gs):,.0f}")
best = max(range(LB, UB + 1), key=profit)
print(f"Grid search over whole dollars: x* = ${best}, profit = ${profit(best):,.0f}")

# Duality and simple operations do not move the optimal decision (Figure 1.4)
fk = lambda x: profit(x) / 1000         # profit in K$
grid = [i / 10 for i in range(0, 2501)]
for name, g in [("base, f", fk), ("duality, -f (minimize)", lambda x: -fk(x)), ("stretched, f/2", lambda x: fk(x) / 2), ("shifted, f + 3", lambda x: fk(x) + 3)]:
    pick = min(grid, key=g) if name.startswith("duality") else max(grid, key=g)
    print(f"{name:24s} best x = {pick:.1f}, value = {g(pick):,.2f} K$")

## 3. Multi-objective EV design
Nondominated designs and weighted-sum choices (Figure 1.6).

In [ ]:
# Multi-objective electric vehicle design (Section 1.3.2, Figure 1.6, Acceleration_vs_EPA.ipynb)
# minimize acceleration time (0-60 mph, s) and maximize EPA range (miles)
designs = [(1.7, 210), (1.9, 270), (2.0, 340), (2.2, 350), (2.4, 355), (2.7, 370), (3.2, 380),
           (3.7, 410), (4.0, 450), (4.5, 470), (5.0, 500), (5.5, 520), (6.0, 530)]


def dominates(p, q):
    """p dominates q if it is no worse in both objectives and better in at least one."""
    return p[0] <= q[0] and p[1] >= q[1] and p != q


pareto = [p for p in designs if not any(dominates(q, p) for q in designs)]
print(f"{len(pareto)} of {len(designs)} designs are nondominated: every gain in range costs acceleration")

# Preference-based approach: normalize both objectives to [0, 1] and scalarize with a weight w on range
a0, a1 = min(d[0] for d in designs), max(d[0] for d in designs)
r0, r1 = min(d[1] for d in designs), max(d[1] for d in designs)
for w in (0.0, 0.25, 0.5, 0.75, 1.0):
    score = lambda d: w * (d[1] - r0) / (r1 - r0) + (1 - w) * (a1 - d[0]) / (a1 - a0)
    best = max(designs, key=score)
    print(f"weight on range {w:.2f}: choose {best[0]} s, {best[1]} miles")

## 4. Hard and soft constraints
The cargo bike loading problem of Figure 1.7, solved by exhaustive search.

In [ ]:
# The cargo bike loading problem: hard vs. soft constraints (Section 1.3.3, Figure 1.7)
from itertools import product

weight = [14.6, 20, 8.5, 10, 13, 9.6, 4.9, 16.5, 8.77, 7.8]          # kg, items 1 to 10
profit = [14.54, 15.26, 5.8, 12.12, 8.2, 7.4, 10.3, 13.5, 6.6, 20.9]  # $
C, PENALTY = 100, 50                                                  # max payload (kg) and overweight penalty


def evaluate(x, soft):
    w = sum(wi for wi, xi in zip(weight, x) if xi)
    p = sum(pi for pi, xi in zip(profit, x) if xi)
    if w > C:
        return (p - PENALTY if soft else float("-inf")), w, p    # hard: infeasible; soft: penalized
    return p, w, p


for soft in (False, True):
    best = max(product((0, 1), repeat=10), key=lambda x: evaluate(x, soft)[0])
    score, w, p = evaluate(best, soft)
    print(f"{'soft' if soft else 'hard'} constraint: items {[i + 1 for i, b in enumerate(best) if b]}, {w:.2f} kg, ${p:.2f}")
everything = evaluate((1,) * 10, True)
print(f"Loading all ten parcels: {everything[1]:.2f} kg, profit ${everything[2]:.2f}, penalized score {everything[0]:.2f}")
print(f"Search space: 2^10 = {2 ** 10} subsets, checked exhaustively")

## 5. State spaces and the search dilemma
The elevator and TSP state counts, and an exploration-exploitation experiment.

In [ ]:
# Ill-structured problems and the search dilemma (Sections 1.4 and 1.5)
import math
import random

# 1. Size of the elevator dispatching state space (Figure 1.9) and of a 13-city TSP
floors, cars = 10, 4
states = floors ** cars * 2 ** (floors * cars) * 2 ** (2 * (floors - 1))
print(f"Elevator states: 10^4 x 2^40 x 2^18 = {states:.3e}")
print(f"13-city asymmetric TSP: 13! = {math.factorial(13):,} tours")

# 2. Exploration vs. exploitation on a landscape with a wide local and a narrow global minimum
def f(x, y):
    return -(8 * math.exp(-((x - 2.2) ** 2 + (y + 1.6) ** 2) / 0.12)
             + 5 * math.exp(-((x + 1.4) ** 2 + (y - 1.2) ** 2) / 2.0)
             + 3.5 * math.exp(-((x - 1.2) ** 2 + (y - 2.1) ** 2) / 0.5))


def search(eps, rng, budget=150, sigma=0.2):
    """With probability eps sample anywhere (explore); otherwise sample near the best point (exploit)."""
    bx, by = rng.uniform(-3, 3), rng.uniform(-3, 3)
    best = f(bx, by)
    for _ in range(budget - 1):
        if rng.random() < eps:
            x, y = rng.uniform(-3, 3), rng.uniform(-3, 3)
        else:
            x = min(3, max(-3, bx + rng.gauss(0, sigma)))
            y = min(3, max(-3, by + rng.gauss(0, sigma)))
        if f(x, y) < best:
            bx, by, best = x, y, f(x, y)
    return best


rng = random.Random(1)
for eps in (0, 0.1, 0.3, 0.5, 0.7, 0.9, 1):
    runs = [search(eps, rng) for _ in range(400)]
    hits = sum(r < -7.6 for r in runs) / len(runs)            # within 5% of the global minimum (-8)
    print(f"exploration rate {eps:.1f}: found the global minimum in {hits:5.1%} of runs")